# Merging centers

This notebook contains my tentative for merging close detected foci.

In [63]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
import numpy as np
import tensorflow as tf
import sys
from scipy.spatial import cKDTree

my_path = Path.cwd().parent.parent / "src"
sys.path.append(str(my_path))

from visualization import visualization 
from utils import utils
from detection import blob_detection
from pairing import pairing

%matplotlib inline

In [3]:
ROOT = Path(r"K:\users\voland\images\data\TgCetnEos_CenSpark_H2B_4h-24h-48hpf\20260506CFHa_CetnEos_H2BmCherry_CS_24hpf\3e1\subsets")

VOL_PATH = ROOT/"s1_20260506CFHa_CetnEos_H2BmCherry_CS_24hpf.lif - 3e1-1.tif"

In [78]:
vol = tifffile.imread(VOL_PATH)
scale = utils.get_pixel_size(VOL_PATH)
vol.shape

(69, 4, 438, 571)

In [84]:
blob_centers = blob_detection.frame_blob_detection(vol[0],"dog",0.1)
blob_centers

array([[  0.        , 102.        , 265.        ,   1.73205081],
       [  0.        , 252.        , 427.        ,   1.73205081],
       [  0.        ,  95.        ,  51.        ,   1.73205081],
       [  0.        ,   9.        , 173.        ,   1.73205081],
       [  0.        , 290.        , 406.        ,   1.73205081],
       [  2.        ,  92.        , 294.        ,   1.73205081],
       [  0.        , 106.        , 315.        ,   1.73205081],
       [  2.        , 342.        , 383.        ,   1.73205081],
       [  1.        , 311.        , 395.        ,   1.73205081],
       [  0.        , 376.        , 371.        ,   1.73205081],
       [  0.        , 196.        , 303.        ,   1.73205081],
       [  1.        , 403.        , 388.        ,   1.73205081],
       [  1.        , 323.        , 372.        ,   1.73205081],
       [  1.        , 400.        , 385.        ,   1.73205081],
       [  1.        , 325.        , 385.        ,   1.73205081],
       [  1.        , 325

Let's get the index of the spots that are closer than a set merging_distance. These spots should be merged as they are too close from each others.

In [85]:
centers_um = blob_centers[:, :3] * scale
merging_distance = 0.7
merging_row, merging_col = pairing.pairing_points_anisotropic(centers_um,centers_um,scale, merging_distance,max_pairing_distance=3,diagonal_pairing=False)

In [10]:
merging_row

array([  8,  11,  14,  15,  18,  19,  20,  21,  25,  29,  30,  32,  34,
        36,  43,  46,  47,  50,  53,  55,  57,  58,  60,  65,  66,  69,
        70,  71,  75,  78,  79,  83,  85,  87,  88,  89,  92,  93,  96,
        97,  99, 101, 102, 109, 113, 115, 122, 124, 126, 127, 129, 132,
       134, 137, 142, 147, 152, 158, 159, 161, 165, 171, 173, 183, 187,
       193, 198, 199])

In [49]:
merging_col

array([ 78,  13,  46, 165, 137, 199,  89, 109,  79,  55,  93,  96, 191,
        58, 198,  14,  66,  88,  69,  29, 115,  36, 193, 102,  47,  53,
       159, 183,  97,   8,  25, 158,  92, 134,  50,  20,  85,  30,  32,
        75, 201, 127,  65, 171, 126,  57, 187, 142, 113, 101,  81, 152,
        87,  18, 124,  34, 132,  83,  70, 195,  15,  21, 172,  71, 122,
        60,  43,  19])

In [41]:
unmerged_centers = blob_centers[~np.isin(np.arange(len(blob_centers)), merging_row)]

In [60]:
merged_centers = np.mean([blob_centers[merging_row],blob_centers[merging_col]],axis=0)
merged_centers = np.unique(merged_centers,axis=0)

In [61]:
new_centers = np.concatenate([unmerged_centers,merged_centers])

In [62]:
new_centers.shape

(173, 4)

Knearest neighbours

In [67]:
tree = cKDTree(centers_um)
pairs = tree.query_pairs(r=merging_distance, output_type='ndarray')  # shape (M, 2)
pairs

array([[103, 126],
       [113, 126],
       [107, 139],
       [ 43, 198],
       [ 83, 158],
       [ 83, 184],
       [ 57, 115],
       [ 18, 184],
       [ 18, 137],
       [101, 127],
       [ 44, 137],
       [ 25, 137],
       [ 25,  79],
       [ 79, 195],
       [161, 195],
       [ 60, 193],
       [ 99, 201],
       [ 81, 129],
       [122, 187],
       [ 29, 162],
       [171, 187],
       [109, 187],
       [109, 171],
       [ 21, 171],
       [ 21, 109],
       [ 29,  55],
       [ 29,  45],
       [ 49, 159],
       [ 70, 159],
       [ 45,  69],
       [  8,  89],
       [  8,  20],
       [  8,  78],
       [ 20,  89],
       [124, 142],
       [ 36,  58],
       [ 17,  58],
       [ 49,  70],
       [ 85,  92],
       [ 46,  85],
       [ 46,  92],
       [ 75,  97],
       [ 50,  88],
       [ 93,  97],
       [ 92, 165],
       [ 14,  46],
       [ 46, 165],
       [ 30,  93],
       [ 37,  77],
       [111, 179],
       [ 47,  66],
       [ 15, 165],
       [ 71,

In [81]:
len(centers_um)

202

In [82]:
len(blob_centers)

202

In [86]:
merged = True
while merged:
    merged = False
    tree = cKDTree(centers_um)
    
    # Find all pairs within merging_distance
    pairs = tree.query_pairs(r=merging_distance, output_type='ndarray')  # shape (M, 2)
    
    if len(pairs) == 0:
        break
    
    unmerged = np.zeros(len(centers_um), dtype=bool)
    merge_map = {}  # idx -> merged result

    for i, j in pairs:
        if unmerged[i] or unmerged[j]:
            continue
        # Merge i and j
        merge_map[i] = (blob_centers[i] + blob_centers[j]) / 2
        unmerged[i] = True
        unmerged[j] = True
        merged = True

    # Keep unmerged spots + new merged spots
    unmerged_centers = blob_centers[~unmerged]
    new_merged = np.array(list(merge_map.values()))
    blob_centers = np.concatenate([unmerged_centers, new_merged]) if len(new_merged) > 0 else unmerged_centers
    centers_um = blob_centers[:, :3] * scale

In [88]:
len(blob_centers)

153